# Segmentacao Semantica - UNet + Oxford-IIIT Pet -> TFLite (LiteRT)

Pipeline completo da atividade:
1. Dataset **Oxford-IIIT Pet** (subset reduzido), trimap como mascara
2. **UNet** treinada do zero (3 classes: pet, fundo, borda)
3. Metricas **IoU** e **Acuracia** com `torchmetrics`
4. Visualizacao da mascara sobreposta
5. Export PyTorch -> `.tflite` (FP32) com `ai-edge-torch`
6. Sanity-check rodando o `.tflite` no proprio Python

> **Como rodar:** habilite GPU em *Runtime > Change runtime type > GPU*.
> Rode as secoes 1 a 4 normalmente. A **secao 5 (export)** pede para
> reiniciar o runtime no meio - as instrucoes estao la.

## Setup
`torch`, `torchvision` e `matplotlib` ja vem no Colab. So falta o `torchmetrics`.

In [ ]:
!pip install torchmetrics -q

## Imports e configuracao

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import OxfordIIITPet
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from torchmetrics import JaccardIndex, Accuracy

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 128       # entrada/saida quadrada -> facilita o lado Android
NUM_CLASSES = 3      # 0=pet, 1=fundo, 2=borda
print("Device:", DEVICE)

## 1. Dataset

O trimap do Oxford Pet tem valores `{1,2,3}`; remapeamos para `{0,1,2}`.
A imagem e normalizada so para `[0,1]` (sem mean/std do ImageNet), pra
manter o pre-processamento no Android simples (basta dividir por 255).

In [ ]:
class PetSegDataset(torch.utils.data.Dataset):
    """Aplica o MESMO resize na imagem e na mascara (nearest p/ mascara)."""
    def __init__(self, base, img_size=IMG_SIZE):
        self.base = base
        self.img_size = img_size

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, trimap = self.base[idx]
        img = TF.resize(img, [self.img_size, self.img_size],
                        interpolation=TF.InterpolationMode.BILINEAR)
        trimap = TF.resize(trimap, [self.img_size, self.img_size],
                           interpolation=TF.InterpolationMode.NEAREST)
        img = TF.to_tensor(img)                                  # [3,H,W] em [0,1]
        mask = torch.from_numpy(np.array(trimap)).long() - 1     # {1,2,3}->{0,1,2}
        mask = mask.clamp(0, NUM_CLASSES - 1)
        return img, mask


def build_loaders(n_train=400, n_val=120, batch_size=8):
    train_base = OxfordIIITPet(root="./data", split="trainval",
                               target_types="segmentation", download=True)
    test_base = OxfordIIITPet(root="./data", split="test",
                              target_types="segmentation", download=True)
    train_ds = Subset(PetSegDataset(train_base), range(min(n_train, len(train_base))))
    val_ds = Subset(PetSegDataset(test_base), range(min(n_val, len(test_base))))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    return train_loader, val_loader

## 2. UNet (compacta, base = 32 canais)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, n_classes=NUM_CLASSES, base=32):
        super().__init__()
        self.d1 = DoubleConv(3, base)
        self.d2 = DoubleConv(base, base * 2)
        self.d3 = DoubleConv(base * 2, base * 4)
        self.bott = DoubleConv(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.u3 = DoubleConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.u2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.u1 = DoubleConv(base * 2, base)
        self.out = nn.Conv2d(base, n_classes, 1)

    def forward(self, x):
        c1 = self.d1(x)
        c2 = self.d2(self.pool(c1))
        c3 = self.d3(self.pool(c2))
        b = self.bott(self.pool(c3))
        x = self.u3(torch.cat([self.up3(b), c3], dim=1))
        x = self.u2(torch.cat([self.up2(x), c2], dim=1))
        x = self.u1(torch.cat([self.up1(x), c1], dim=1))
        return self.out(x)        # logits [N,3,H,W]

## 3. Treino + metricas (IoU / Acuracia)

In [ ]:
def train(epochs=15, lr=1e-3):
    train_loader, val_loader = build_loaders()
    model = UNet().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    iou = JaccardIndex(task="multiclass", num_classes=NUM_CLASSES).to(DEVICE)
    acc = Accuracy(task="multiclass", num_classes=NUM_CLASSES).to(DEVICE)

    for ep in range(1, epochs + 1):
        model.train()
        running = 0.0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(imgs), masks)
            loss.backward()
            opt.step()
            running += loss.item() * imgs.size(0)
        train_loss = running / len(train_loader.dataset)

        model.eval()
        iou.reset(); acc.reset()
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                preds = model(imgs).argmax(dim=1)
                iou.update(preds, masks)
                acc.update(preds, masks)
        print(f"Epoch {ep:02d} | loss={train_loss:.4f} "
              f"| IoU={iou.compute().item():.4f} | Acc={acc.compute().item():.4f}")
    return model, val_loader

In [ ]:
model, val_loader = train(epochs=15)

# Salva os pesos: vamos recarregar depois de reiniciar o runtime (secao 5).
torch.save(model.state_dict(), "unet_pet.pth")
print("Pesos salvos em unet_pet.pth")

## 4. Visualizacao da mascara sobreposta

In [ ]:
def visualize(model, val_loader, n=4):
    model.eval()
    imgs, masks = next(iter(val_loader))
    with torch.no_grad():
        preds = model(imgs.to(DEVICE)).argmax(dim=1).cpu()
    palette = np.array([[255, 0, 0], [0, 0, 0], [0, 255, 0]], dtype=np.uint8)  # pet/fundo/borda
    fig, ax = plt.subplots(n, 3, figsize=(9, 3 * n))
    for i in range(n):
        img = imgs[i].permute(1, 2, 0).numpy()
        gt = palette[masks[i].numpy()]
        pr = palette[preds[i].numpy()]
        overlay = (0.6 * img + 0.4 * (pr / 255.0)).clip(0, 1)
        ax[i, 0].imshow(img);     ax[i, 0].set_title("Original");     ax[i, 0].axis("off")
        ax[i, 1].imshow(gt);      ax[i, 1].set_title("Ground truth"); ax[i, 1].axis("off")
        ax[i, 2].imshow(overlay); ax[i, 2].set_title("Predicao");     ax[i, 2].axis("off")
    plt.tight_layout()
    plt.savefig("resultados.png", dpi=120)
    plt.show()

visualize(model, val_loader)

## 5. Export para TFLite (FP32) com ai-edge-torch

O `ai-edge-torch` instala uma versao especifica do torch, entao o fluxo seguro e:

1. Rode a celula de **instalacao** abaixo.
2. **Reinicie o runtime:** *Runtime > Restart session*.
3. Rode novamente **so** a celula de *Imports e configuracao* (secao topo) e a
   celula da **classe UNet** (secao 2) - NAO precisa retreinar.
4. Rode as duas celulas de export abaixo. Elas recarregam os pesos de `unet_pet.pth`.

> **NCHW:** o `ai-edge-torch` preserva a assinatura do PyTorch, entao o `.tflite`
> tera entrada e saida `[1, 3, 128, 128]`. Isso importa no lado Android.

In [ ]:
!pip install ai-edge-torch ai-edge-litert -q
print('Agora: Runtime > Restart session, depois rode os imports + a classe UNet de novo.')

**Export** (rode depois do restart + re-rodar imports e a classe UNet):

In [ ]:
import ai_edge_torch

model = UNet()
model.load_state_dict(torch.load("unet_pet.pth", map_location="cpu"))
model.eval()

sample = (torch.randn(1, 3, IMG_SIZE, IMG_SIZE),)
edge_model = ai_edge_torch.convert(model, sample)
edge_model.export("unet_pet.tflite")
print("Salvo unet_pet.tflite -", round(os.path.getsize("unet_pet.tflite") / 1e6, 2), "MB")

## 6. Sanity-check: roda o `.tflite` e compara com o PyTorch

In [ ]:
from ai_edge_litert.interpreter import Interpreter

interp = Interpreter(model_path="unet_pet.tflite")
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print("input :", inp["shape"], inp["dtype"])
print("output:", out["shape"], out["dtype"])

x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
with torch.no_grad():
    torch_out = model(x).argmax(1).numpy()
interp.set_tensor(inp["index"], x.numpy().astype(np.float32))
interp.invoke()
tfl_out = interp.get_tensor(out["index"]).argmax(1)
print(f"Concordancia PyTorch vs TFLite: {(torch_out == tfl_out).mean()*100:.2f}%")

## 7. Baixar os artefatos

In [ ]:
from google.colab import files
files.download("unet_pet.tflite")
files.download("resultados.png")